# 📂 Data Splitting Notebook

**Purpose:** Create reproducible K-Fold splits to prevent patient-level data leakage.

**Output:** CSV files for each fold saved to `splits/` directory.

---

**Key Features:**
- GroupKFold by `patient_id` (no patient overlap between train/val)
- Preserves all mask/label entries per image (handles mixed pathology)
- Serialized mask paths for easy loading

# 0. Download and unzip the dataset

In [1]:
# ═══════════════════════════════════════════════════════════════════════════
# DOWNLOAD AND EXTRACT DATA
# ═══════════════════════════════════════════════════════════════════════════
import gdown
import zipfile
import os

# Download data
url = "https://drive.google.com/drive/folders/1uTP8xcNqwi2MYUptg3BZwGjjDOMFhNBC?usp=sharing"
gdown.download_folder(url, output="breast_ultrasound_data", quiet=False)

# Unzip images
zip_path = "breast_ultrasound_data/Training_Images.zip"
extract_to = "breast_ultrasound_data/training_images"

os.makedirs(extract_to, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print(f"✓ Unzipped files: {len(os.listdir(extract_to))}")

Retrieving folder contents


Processing file 1VGohBSMxPSyxJm912O9O-x6fmA-buoSE Training_Images.zip
Processing file 1Oxbq4hGIyU6No9BCiQSsyXL1Dz41R-NU training_metadata.xlsx


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1VGohBSMxPSyxJm912O9O-x6fmA-buoSE
From (redirected): https://drive.google.com/uc?id=1VGohBSMxPSyxJm912O9O-x6fmA-buoSE&confirm=t&uuid=360498ef-02ed-434c-94b0-2ce7046e365b
To: /content/breast_ultrasound_data/Training_Images.zip
100%|██████████| 582M/582M [00:09<00:00, 62.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Oxbq4hGIyU6No9BCiQSsyXL1Dz41R-NU
To: /content/breast_ultrasound_data/training_metadata.xlsx
100%|██████████| 43.9k/43.9k [00:00<00:00, 52.4MB/s]
Download completed


✓ Unzipped files: 1


## 1. Configuration & Imports

In [2]:
import os
import pandas as pd
from sklearn.model_selection import GroupKFold

# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

METADATA_PATH = "breast_ultrasound_data/training_metadata.xlsx"
IMAGE_FOLDER = "breast_ultrasound_data/training_images/training_images"
OUTPUT_DIR = "splits"
N_SPLITS = 5

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Configuration:")
print(f"  Metadata: {METADATA_PATH}")
print(f"  Images: {IMAGE_FOLDER}")
print(f"  Output: {OUTPUT_DIR}/")
print(f"  K-Folds: {N_SPLITS}")

Configuration:
  Metadata: breast_ultrasound_data/training_metadata.xlsx
  Images: breast_ultrasound_data/training_images/training_images
  Output: splits/
  K-Folds: 5


## 2. Load Raw Metadata

In [3]:
# ═══════════════════════════════════════════════════════════════════════════
# LOAD RAW METADATA
# ═══════════════════════════════════════════════════════════════════════════

df_raw = pd.read_excel(
    METADATA_PATH,
    header=None,
    names=["image_filename", "mask_filename", "label"]
)

# Skip header row if present
df_raw = df_raw.iloc[1:].reset_index(drop=True)

# Extract patient ID from filename
def extract_patient_id(filename):
    """Extract patient ID from filename (e.g., '123_0.png' -> 123)"""
    base = filename.replace(".png", "")
    if "_" in base:
        return int(base.split("_")[0])
    return int(base)

df_raw["patient_id"] = df_raw["image_filename"].apply(extract_patient_id)

# Verify image exists
df_raw["image_exists"] = df_raw["image_filename"].apply(
    lambda x: os.path.exists(os.path.join(IMAGE_FOLDER, x))
)
df_raw = df_raw[df_raw["image_exists"]].reset_index(drop=True)
df_raw = df_raw.drop(columns=["image_exists"])

# Labels are numeric (0=Benign, 1=Malignant, 2=Normal)
df_raw['label'] = pd.to_numeric(df_raw['label'], errors='coerce').astype('Int64')

# Drop invalid labels
before_drop = len(df_raw)
df_raw = df_raw.dropna(subset=['label']).reset_index(drop=True)
after_drop = len(df_raw)
if before_drop > after_drop:
    print(f"⚠️  Dropped {before_drop - after_drop} rows with invalid labels")

df_raw['label'] = df_raw['label'].astype(int)

print(f"\n{'='*60}")
print("RAW DATASET LOADED")
print(f"{'='*60}")
print(f"Total entries: {len(df_raw)}")
print(f"Unique patients: {df_raw['patient_id'].nunique()}")
print(f"Unique images: {df_raw['image_filename'].nunique()}")
print(f"\nLabel distribution (0=Benign, 1=Malignant, 2=Normal):")
print(df_raw['label'].value_counts().sort_index())


RAW DATASET LOADED
Total entries: 1503
Unique patients: 968
Unique images: 1503

Label distribution (0=Benign, 1=Malignant, 2=Normal):
label
0    679
1    364
2    460
Name: count, dtype: int64


## 3. Group Mask Filenames by Image

Some images have multiple masks (mixed pathology). We group them and store as a semicolon-separated string for CSV serialization.

In [4]:
# ═══════════════════════════════════════════════════════════════════════════
# GROUP MASKS BY IMAGE (Handle Mixed Pathology)
# ═══════════════════════════════════════════════════════════════════════════

df_grouped = df_raw.groupby('image_filename').agg({
    'mask_filename': lambda x: ';'.join(x),  # Serialize list as semicolon-separated string
    'label': 'max',  # Worst-case label (Malignant > Benign > Normal)
    'patient_id': 'first'  # Keep patient ID
}).reset_index()

# Rename for clarity
df_grouped = df_grouped.rename(columns={'mask_filename': 'mask_filenames'})

print(f"Grouped dataset: {len(df_grouped)} unique images")
print(f"Mask serialization example: {df_grouped['mask_filenames'].iloc[0]}")

Grouped dataset: 1503 unique images
Mask serialization example: 1_mask.png


## 4. K-Fold Splitting Loop

In [5]:
# ═══════════════════════════════════════════════════════════════════════════
# K-FOLD SPLITTING
# ═══════════════════════════════════════════════════════════════════════════

gkf = GroupKFold(n_splits=N_SPLITS)

# Storage for fold statistics
fold_stats = []

print(f"\n{'='*70}")
print(f"GENERATING {N_SPLITS}-FOLD SPLITS (GroupKFold by patient_id)")
print(f"{'='*70}\n")

for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(df_grouped, groups=df_grouped['patient_id'])):

    print(f"{'─'*70}")
    print(f"FOLD {fold_idx}")
    print(f"{'─'*70}")

    # ═══════════════════════════════════════════════════════════════════════
    # STEP A: Split raw dataframe
    # ═══════════════════════════════════════════════════════════════════════
    train_df = df_grouped.iloc[train_idx].reset_index(drop=True)
    val_df = df_grouped.iloc[val_idx].reset_index(drop=True)

    print(f"\n  Step A - Split by Patient:")
    print(f"    Train: {len(train_df)} images, {train_df['patient_id'].nunique()} patients")
    print(f"    Val:   {len(val_df)} images, {val_df['patient_id'].nunique()} patients")

    # Verify no patient overlap
    train_patients = set(train_df['patient_id'].unique())
    val_patients = set(val_df['patient_id'].unique())
    overlap = train_patients & val_patients
    assert len(overlap) == 0, f"Patient leak detected! Overlapping patients: {overlap}"
    print(f"    ✓ No patient overlap")

    # ═══════════════════════════════════════════════════════════════════════
    # STEP B: Save to CSV
    # ═══════════════════════════════════════════════════════════════════════
    train_path = os.path.join(OUTPUT_DIR, f"fold_{fold_idx}_train.csv")
    val_path = os.path.join(OUTPUT_DIR, f"fold_{fold_idx}_val.csv")

    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path, index=False)

    print(f"\n  Step B - Saved:")
    print(f"    Train: {train_path}")
    print(f"    Val:   {val_path}")

    # ═══════════════════════════════════════════════════════════════════════
    # VERIFICATION: Print final statistics
    # ═══════════════════════════════════════════════════════════════════════
    stats = {
        'fold': fold_idx,
        'train_images': len(train_df),
        'train_patients': train_df['patient_id'].nunique(),
        'val_images': len(val_df),
        'val_patients': val_df['patient_id'].nunique(),
        'train_label_dist': dict(train_df['label'].value_counts().sort_index()),
        'val_label_dist': dict(val_df['label'].value_counts().sort_index())
    }
    fold_stats.append(stats)

    print(f"\n  📊 FOLD {fold_idx} SUMMARY:")
    print(f"    ┌{'─'*50}┐")
    print(f"    │ {'Split':<10} │ {'Images':>10} │ {'Patients':>10} │ {'Labels (0/1/2)':>15} │")
    print(f"    ├{'─'*50}┤")
    print(f"    │ {'Train':<10} │ {len(train_df):>10} │ {train_df['patient_id'].nunique():>10} │ {str(stats['train_label_dist']):>15} │")
    print(f"    │ {'Val':<10} │ {len(val_df):>10} │ {val_df['patient_id'].nunique():>10} │ {str(stats['val_label_dist']):>15} │")
    print(f"    └{'─'*50}┘")
    print()


GENERATING 5-FOLD SPLITS (GroupKFold by patient_id)

──────────────────────────────────────────────────────────────────────
FOLD 0
──────────────────────────────────────────────────────────────────────

  Step A - Split by Patient:
    Train: 1202 images, 774 patients
    Val:   301 images, 194 patients
    ✓ No patient overlap

  Step B - Saved:
    Train: splits/fold_0_train.csv
    Val:   splits/fold_0_val.csv

  📊 FOLD 0 SUMMARY:
    ┌──────────────────────────────────────────────────┐
    │ Split      │     Images │   Patients │  Labels (0/1/2) │
    ├──────────────────────────────────────────────────┤
    │ Train      │       1202 │        774 │ {0: np.int64(540), 1: np.int64(279), 2: np.int64(383)} │
    │ Val        │        301 │        194 │ {0: np.int64(139), 1: np.int64(85), 2: np.int64(77)} │
    └──────────────────────────────────────────────────┘

──────────────────────────────────────────────────────────────────────
FOLD 1
──────────────────────────────────────────────

## 5. Overall Summary

In [6]:
# ═══════════════════════════════════════════════════════════════════════════
# OVERALL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════

print(f"\n{'='*70}")
print("K-FOLD SPLITTING COMPLETE")
print(f"{'='*70}\n")

summary_df = pd.DataFrame(fold_stats)
summary_df = summary_df[['fold', 'train_images', 'train_patients', 'val_images', 'val_patients']]

print("Fold Statistics:")
print(summary_df.to_string(index=False))

print(f"\n\nOutput Files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  ✓ {f} ({size_kb:.1f} KB)")

print(f"\n\n✅ All {N_SPLITS} folds saved to: {OUTPUT_DIR}/")


K-FOLD SPLITTING COMPLETE

Fold Statistics:
 fold  train_images  train_patients  val_images  val_patients
    0          1202             774         301           194
    1          1202             775         301           193
    2          1202             774         301           194
    3          1203             773         300           195
    4          1203             776         300           192


Output Files:
  ✓ fold_0_train.csv (35.1 KB)
  ✓ fold_0_val.csv (8.8 KB)
  ✓ fold_1_train.csv (35.1 KB)
  ✓ fold_1_val.csv (8.8 KB)
  ✓ fold_2_train.csv (35.1 KB)
  ✓ fold_2_val.csv (8.8 KB)
  ✓ fold_3_train.csv (35.1 KB)
  ✓ fold_3_val.csv (8.8 KB)
  ✓ fold_4_train.csv (35.1 KB)
  ✓ fold_4_val.csv (8.8 KB)


✅ All 5 folds saved to: splits/


## 6. Verification: Load and Parse a Saved Split

In [7]:
# ═══════════════════════════════════════════════════════════════════════════
# VERIFICATION: Load and parse a saved split
# ═══════════════════════════════════════════════════════════════════════════

print("Verification: Loading fold_0_train.csv...\n")

test_df = pd.read_csv(os.path.join(OUTPUT_DIR, "fold_0_train.csv"))

# Parse mask_filenames back to list
test_df['mask_filenames_list'] = test_df['mask_filenames'].apply(lambda x: x.split(';'))

print("Columns:", list(test_df.columns))
print(f"\nFirst 3 rows:")
print(test_df[['image_filename', 'mask_filenames', 'label', 'patient_id']].head(3))

print(f"\nParsed mask_filenames_list example:")
print(f"  {test_df['mask_filenames_list'].iloc[0]}")

print(f"\n✓ CSV loading and parsing verified successfully!")

Verification: Loading fold_0_train.csv...

Columns: ['image_filename', 'mask_filenames', 'label', 'patient_id', 'mask_filenames_list']

First 3 rows:
  image_filename mask_filenames  label  patient_id
0          1.png     1_mask.png      2           1
1         10.png    10_mask.png      2          10
2        100.png   100_mask.png      2         100

Parsed mask_filenames_list example:
  ['1_mask.png']

✓ CSV loading and parsing verified successfully!


---

## 📋 Usage in Training Notebooks

To load a specific fold in the training notebook:

```python
import pandas as pd

FOLD = 0  # Change this to use different folds

train_df = pd.read_csv(f'splits/fold_{FOLD}_train.csv')
val_df = pd.read_csv(f'splits/fold_{FOLD}_val.csv')

# Parse mask_filenames back to list
train_df['mask_filenames_list'] = train_df['mask_filenames'].apply(lambda x: x.split(';'))
val_df['mask_filenames_list'] = val_df['mask_filenames'].apply(lambda x: x.split(';'))
```